In [89]:

import pandas as pd
from opencc import OpenCC
import numpy as np
import os


inputdirectory = '../../50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements/'
globalinputdirectory = '../../50 KM Group/Royalties/Statements/Karen/All labels combined/lookup_tables/'
outputdirectory = '../../50 KM Group/Royalties/Statements/Karen/_output/'

file_Rock = 'Rock_royalties_2021Q1_2026Q1_1_raw_combined.csv' 

lookup_fx = 'lookup_tables/Rock_lookup_fx.csv'
lookup_isrc = 'lookup_tables/Rock_lookup_ISRC_song_album_albumtype.xlsx'
#lookup_album = 'lookup_tables/Rock_lookup_album.csv'
#lookup_song = 'lookup_tables/Rock_lookup_ISRC_song_album_albumtype.csv'


outputfile = 'Rock_royalties_2021Q1_2026Q1_2_matched.csv'
#outputfilexls = 'Rock_royalties_2021Q1_2025Q3_2_matched.xlsx'
converter = OpenCC('s2t') 

def readfile(directory,file):
    path = os.path.join(directory, file)
    df = pd.read_csv(path,low_memory=False)
    print(f"The dataframe of the file '{file}' has {df.shape[0]} rows and {len(df.columns)} columns.")
    return df

def readfilexls(directory,file,sheet):
    path = os.path.join(directory, file)
    df = pd.read_excel(path,sheet_name=sheet)
    print(f"The dataframe of the file '{file}' (tab '{sheet}') has {df.shape[0]} rows and {len(df.columns)} columns.")
    return df

def merge(df1, df2, col, what):
    print(f"\nMerging '{what}' on '{col}':")
    empty_cells = df1[col].isna().sum()
    print(f"There are a total of {empty_cells} rows that have no entry in {col}")
    df1.loc[:, col] = df1[col].fillna('XX_UNKNOWN')
    #df1.fillna({col: 'n/a'}, inplace=True)
    df_merged = pd.merge(df1, df2, on=col, how='left', indicator='_merge_status')
    unmatched_mask = df_merged['_merge_status'] == 'left_only'
    n_unmatched = unmatched_mask.sum()
    df_merged = df_merged.drop(columns='_merge_status')
    if n_unmatched == 0:
        print(f"Merging of column {col} was successful")
    else:
        print(f"Merging with issues. There are a total of {n_unmatched} rows that could not be matched (see 'match_issues_{what}_{col}.xlsx').")
        empty_rows = df_merged.loc[unmatched_mask]
        col_safe = col.replace('/','')
        empty_rows.to_csv(f"{outputdirectory}match_issues_{what}_{col_safe}.csv", index=False)
        #empty_rows.to_csv(f"{outputdirectory}match_issues_{what}_{col_safe}.csv", engine='openpyxl', index=False)
    unused_rows = df2[~df2[col].isin(df_merged[col])]
    if len(unused_rows) > 0:
        unused_rows.to_csv(f'{outputdirectory}unused_lookup_rows_{col}.csv', index=False)  
    print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")
    return df_merged 



def merge_and_return_empty_rows(df1, df2, col, what):
    print(f"\nMerging '{what}' on '{col}':")
    empty_cells = df1[col].isna().sum()
    print(f"There are a total of {empty_cells} rows that have no entry in {col}")
    df1.loc[:, col] = df1[col].fillna('XX_UNKNOWN')
    #df1.fillna({col: 'n/a'}, inplace=True)
    df_merged = pd.merge(df1, df2, on=col, how='left', indicator='_merge_status')
    unmatched_mask = df_merged['_merge_status'] == 'left_only'
    n_unmatched = unmatched_mask.sum()
    empty_rows = df_merged.loc[unmatched_mask].drop(columns='_merge_status')
    df_merged = df_merged.drop(columns='_merge_status')
    if n_unmatched == 0:
        print(f"Merging of column {col} was successful")
    else:
        print(f"Merging with issues. There are a total of {n_unmatched} rows that could not be matched (see 'match_issues_{what}_{col}.xlsx').")
        col_safe = col.replace('/','')
        empty_rows.to_csv(f"{outputdirectory}match_issues_{what}_{col_safe}.csv", index=False)
        #empty_rows.to_csv(f"{outputdirectory}match_issues_{what}_{col_safe}.csv", engine='openpyxl', index=False)
    unused_rows = df2[~df2[col].isin(df_merged[col])]
    if len(unused_rows) > 0:
        unused_rows.to_csv(f'{outputdirectory}unused_lookup_rows_{col}.csv', index=False)  
    print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")
    return df_merged, empty_rows

df_Rock = readfile(inputdirectory, file_Rock)

df_lookup_fx = readfile(inputdirectory, lookup_fx)
df_lookup_isrc = readfilexls(inputdirectory ,lookup_isrc,"Data_MOD")
#df_lookup_album = readfile(inputdirectory, lookup_album)
#df_lookup_song = readfile(globalinputdirectory, lookup_song)
#df_lookup_isrc['ISRC (final)'] = df_lookup_isrc['ISRC (final)'].astype(str)
#df_lookup_song['ISRC (final)'] = df_lookup_song['ISRC (final)'].astype(str)

The dataframe of the file 'Rock_royalties_2021Q1_2026Q1_1_raw_combined.csv' has 1442752 rows and 29 columns.
The dataframe of the file 'lookup_tables/Rock_lookup_fx.csv' has 182 rows and 13 columns.
The dataframe of the file 'lookup_tables/Rock_lookup_ISRC_song_album_albumtype.xlsx' (tab 'Data_MOD') has 5243 rows and 5 columns.


In [90]:
# pull in the FX rates for the local curreny and compute royalties in HKD and the amount to Rock in HKD
df_Rock_fx = pd.merge(df_Rock, df_lookup_fx[['Report year', 'Report Quarter', 'Currency', 'FX rate']],
                     on=['Report year', 'Report Quarter', 'Currency'],
                     how='left')

df_Rock_fx['SHARE AMOUNT (local FX)'] = pd.to_numeric(df_Rock_fx['SHARE AMOUNT (local FX)'], errors="coerce") # new line 05 Sep 2025
df_Rock_fx['AMOUNT (HKD)']=df_Rock_fx['SHARE AMOUNT (local FX)']/df_Rock_fx['FX rate']
df_Rock_fx['Amount to Rock (HKD)'] = np.where(
    df_Rock_fx['AMOUNT (HKD)'] != 0,  # Condition
    df_Rock_fx['AMOUNT'] * df_Rock_fx['AMOUNT (HKD)'] / df_Rock_fx['SHARE AMOUNT (local FX)'],  # True case
    0  # False case
)

# pull in the FX rates for RMB and compute the amount to Rock in HKD
df_Rock_fx.rename(columns={'FX rate': 'FX rate_main'}, inplace=True)
df_lookup_fx_rmb = df_lookup_fx[df_lookup_fx['Currency'] == 'RMB']
df_Rock_fx = pd.merge(df_Rock_fx, df_lookup_fx_rmb[['Report year', 'Report Quarter', 'FX rate']],
                     on=['Report year', 'Report Quarter'],
                     how='left')
df_Rock_fx.rename(columns={'FX rate': 'FX rate_rmb','FX rate_main': 'FX rate'}, inplace=True)
df_Rock_fx['Amount to Rock (RMB)']=df_Rock_fx['Amount to Rock (HKD)']*df_Rock_fx['FX rate_rmb']

# Checking for empty cells in to be matched columns
empty_cells_p = df_Rock_fx['USER'].isna().sum()    
empty_cells_a = df_Rock_fx['CATALOG TITLE'].isna().sum()
empty_cells_s = df_Rock_fx['SONG TITLE'].isna().sum()
print(f"Empty cells in 'USER', 'CATALOG TITLE', 'SONG TITLE': {empty_cells_p} / {empty_cells_a} / {empty_cells_s}")
df_Rock_fx.fillna({'USER':'XX_UNKNOWN','CATALOG TITLE':'Song Type','SONG TITLE':'EMPTY'}, inplace=True)
print(f"The dataframe now has {df_Rock_fx.shape[0]} rows and {len(df_Rock_fx.columns)} columns.")



Empty cells in 'USER', 'CATALOG TITLE', 'SONG TITLE': 0 / 0 / 455
The dataframe now has 1442752 rows and 34 columns.


In [91]:
# matching catalog no . Catalog Title . Song Title 

df_Rock_fx['CATALOG NO.'] = df_Rock_fx['CATALOG NO.'].astype('string')
df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE'] = df_Rock_fx['CATALOG NO._MOD'] + '.' + df_Rock_fx['CATALOG TITLE'] + '.' + df_Rock_fx['SONG TITLE']

df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE'] = df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE'].astype(str)

df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'] = (df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE']
            .str.replace(" ", '', regex=False)
            .str.replace("(", '', regex=False)
            .str.replace(")", '', regex=False)
            .str.replace("-", '', regex=False)
            .str.replace("/", '', regex=False)
            .str.replace('（', '', regex=False)
            .str.replace('）', '', regex=False)
            .str.replace('’', '', regex=False)
            .str.replace("'", '', regex=False)
            .str.lower())

df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'] = df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'].apply(converter.convert)

df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'] = (df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'].str.replace('.empty', '.', regex=False))


df_Rock_matched, unmapped_rows = merge_and_return_empty_rows(df_Rock_fx, df_lookup_isrc, 'CATALOG NO..CATALOG TITLE.SONG TITLE_MOD','Rock_fx')

df_Rock_matched=df_Rock_matched.drop(columns=['CATALOG NO..CATALOG TITLE.SONG TITLE'])
df_Rock_matched=df_Rock_matched.drop(columns=['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'])
df_Rock_matched=df_Rock_matched.drop(columns=['CATALOG NO._MOD'])
#df_Rock_matched = df_Rock_matched.rename(columns={'Album': 'Album (final)'})
#df_Rock_matched = df_Rock_matched.rename(columns={'Song': 'Song (final)'})



df_Rock_matched = df_Rock_matched.sort_index(axis=1)
#path = os.path.join(outputdirectory, outputfilexls)
#df_Rock_matched.to_excel(path, engine='openpyxl', index=True)




Merging 'Rock_fx' on 'CATALOG NO..CATALOG TITLE.SONG TITLE_MOD':
There are a total of 0 rows that have no entry in CATALOG NO..CATALOG TITLE.SONG TITLE_MOD
Merging with issues. There are a total of 34240 rows that could not be matched (see 'match_issues_Rock_fx_CATALOG NO..CATALOG TITLE.SONG TITLE_MOD.xlsx').
The new dataframe has 1442752 rows and 40 columns.


In [92]:
all_isrc = readfilexls(inputdirectory ,lookup_isrc,"lookup")

def normalize_catalog_title(title):
    s = (str(title)
         .replace(" ", '')
         .replace("(", '')
         .replace(")", '')
         .replace("-", '')
         .replace("/", '')
         .replace('（', '')
         .replace('）', '')
         .replace('’', '')
         .replace('《', '')
         .replace('》', '')
         .replace('`', '')
         .replace("'", ''))
    return converter.convert(s.lower())

isrc_to_song = {}
song_to_isrc = {}
song_norm_to_isrc = {}
song_norm_to_song = {}
for index, row in all_isrc.iterrows():
    isrc = row['ISRC']
    song = row['Song (final)']
    isrc_to_song[isrc] = song
    song_to_isrc[song] = isrc
    norm_song = normalize_catalog_title(song)
    song_norm_to_isrc[norm_song] = isrc
    song_norm_to_song[norm_song] = song

all_songs = df_Rock_matched['SONG TITLE'].unique()

all_titles = readfilexls(inputdirectory, lookup_isrc, "Data_MOD")

title_to_album = {}
title_to_albumtype = {}
album_to_albumtype = {}
def extract_title_key(mod_string):
    parts = str(mod_string).split('.')
    if len(parts) < 3:
        raw_title = str(mod_string)
    else:
        raw_title = '.'.join(parts[2:-1])
    return normalize_catalog_title(raw_title)[:10]

for index, row in all_titles.iterrows():
    title = extract_title_key(row['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'])
    album = row['Album']
    albumtype = row['Album Type']
    title_to_album[title] = album
    title_to_albumtype[title] = albumtype
    album_to_albumtype[album] = albumtype

The dataframe of the file 'lookup_tables/Rock_lookup_ISRC_song_album_albumtype.xlsx' (tab 'lookup') has 313 rows and 6 columns.
The dataframe of the file 'lookup_tables/Rock_lookup_ISRC_song_album_albumtype.xlsx' (tab 'Data_MOD') has 5243 rows and 5 columns.


In [93]:
UNFILLED_MARKERS = {'XX_UNKNOWN', 'XX_UNKNOWN'}

def count_filled(before_df, after_df, columns=('ISRC', 'Song')):
    """Count cells filled by the algorithm (empty before, real value after)."""
    cols = list(columns)
    was_empty = before_df[cols].isna() | before_df[cols].isin(UNFILLED_MARKERS)
    total_rows = was_empty.any(axis=1).sum()  # rows still unmapped entering this pass, not len(before_df)
    now_filled = after_df[cols].notna() & ~after_df[cols].isin(UNFILLED_MARKERS)
    filled = was_empty & now_filled

    per_column = filled.sum()
    rows_with_any = filled.any(axis=1).sum()

    print(f"Unmapped rows: {total_rows:,}")
    print("Values filled by algorithm:")
    for col in cols:
        pct = (per_column[col] / total_rows * 100) if total_rows else 0
        print(f"  {col}: {per_column[col]:,} ({pct:.1f}%)")
    rows_pct = (rows_with_any / total_rows * 100) if total_rows else 0
    print(f"Rows with filled value in {cols}: {rows_with_any:,} ({rows_pct:.1f}%)")

    return per_column, rows_with_any


def normalize_for_match(text):
    s = str(text).lower()
    for ch in " `'\"()-（）[]":
        s = s.replace(ch, '')
    return s


def find_subsequence_match(text, candidates, normalizer=None):
    """Return the first candidate whose characters appear in order within text."""
    norm_fn = normalizer or (lambda x: str(x).lower())
    char_list = list(norm_fn(text))
    for candidate in candidates:
        idx = 0
        matched = True
        for char in norm_fn(candidate):
            while idx < len(char_list) and char_list[idx] != char:
                idx += 1
            if idx >= len(char_list):
                matched = False
                break
            idx += 1
        if matched:
            return candidate
    return None


def find_isrc_for_song(matched_song, song_to_isrc):
    """Find ISRC for a matched song via exact, forward, or reverse subsequence match."""
    if matched_song in song_to_isrc:
        return song_to_isrc[matched_song]

    lookup_song = find_subsequence_match(matched_song, song_to_isrc.keys(), normalize_for_match)
    if lookup_song is not None:
        return song_to_isrc[lookup_song]

    for lookup_song in song_to_isrc:
        if find_subsequence_match(lookup_song, [matched_song], normalize_for_match):
            return song_to_isrc[lookup_song]

    return None

def fill_rows_album(df):
    rows_filled = df.copy()
    rows_filled['Album'] = rows_filled['Album'].astype('object')
    rows_filled['Album Type'] = rows_filled['Album Type'].astype('object')
    for index, row in rows_filled.iterrows():
        title_key = extract_title_key(row['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'])
        album_empty = pd.isna(rows_filled.at[index, 'Album']) or rows_filled.at[index, 'Album'] == 'XX_UNKNOWN'
        albumtype_empty = pd.isna(rows_filled.at[index, 'Album Type']) or rows_filled.at[index, 'Album Type'] == 'XX_UNKNOWN'

        if album_empty and title_key in title_to_album:
            rows_filled.at[index, 'Album'] = title_to_album[title_key]
        if albumtype_empty and title_key in title_to_albumtype:
            rows_filled.at[index, 'Album Type'] = title_to_albumtype[title_key]
    return rows_filled


def fill_rows_ISRC_song(df):
    rows_filled = df.copy()
    for index, row in rows_filled.iterrows():
        isrc_number = row['CATALOG NO.']

        if isrc_number in isrc_to_song:
            song_name = isrc_to_song[isrc_number]
            rows_filled.at[index, 'ISRC'] = isrc_number
            rows_filled.at[index, 'Song'] = song_name
        else:
            song = row['SONG TITLE']
            if song in song_to_isrc:
                rows_filled.at[index, 'ISRC'] = song_to_isrc[song]
                rows_filled.at[index, 'Song'] = song
            else:
                norm_song = normalize_catalog_title(song)
                if norm_song in song_norm_to_isrc:
                    rows_filled.at[index, 'ISRC'] = song_norm_to_isrc[norm_song]
                    rows_filled.at[index, 'Song'] = song_norm_to_song[norm_song]
                else:
                    rows_filled.at[index, 'Song'] = 'XX_UNKNOWN'
                    rows_filled.at[index, 'ISRC'] = 'XX_UNKNOWN'
    return rows_filled


adapted_df = fill_rows_ISRC_song(unmapped_rows)

print("\nISRC / Song fill results:")
count_filled(unmapped_rows, adapted_df)

adapted_df = fill_rows_album(adapted_df)

print("\nAlbum / Album Type fill results:")
count_filled(unmapped_rows, adapted_df, columns=('Album', 'Album Type'))

#song_without_isrc = adapted_df[adapted_df['ISRC'] == 'XX_UNKNOWN']
#print(f"\nMatch did not work: {len(song_without_isrc):,} rows")


def fill_rows_ISRC_song_ignore_lyric_mv(df):
    """Attempt: ignore a '歌詞版mv' (lyric video) suffix in the song title before matching."""
    rows_filled = df.copy()
    for index, row in rows_filled.iterrows():
        if row['ISRC'] != 'XX_UNKNOWN':
            continue
        norm_song = normalize_catalog_title(row['SONG TITLE']).replace('歌詞版mv', '')
        if norm_song in song_norm_to_isrc:
            rows_filled.at[index, 'ISRC'] = song_norm_to_isrc[norm_song]
            rows_filled.at[index, 'Song'] = song_norm_to_song[norm_song]
    return rows_filled


before_lyric_mv_pass = adapted_df.copy()
adapted_df = fill_rows_ISRC_song_ignore_lyric_mv(adapted_df)

print("\nLyric-video (歌詞版MV) suffix pass fill results:")
count_filled(before_lyric_mv_pass, adapted_df, columns=('ISRC', 'Song'))

song_norm7_to_isrc = {}
song_norm7_to_song = {}
for isrc, song in isrc_to_song.items():
    key7 = normalize_catalog_title(song)[:7]
    song_norm7_to_isrc[key7] = isrc
    song_norm7_to_song[key7] = song


def fill_rows_ISRC_song_7char(df):
    """Second-pass attempt: match on the first 7 characters of the normalized song title."""
    rows_filled = df.copy()
    for index, row in rows_filled.iterrows():
        if row['ISRC'] != 'XX_UNKNOWN':
            continue
        key7 = normalize_catalog_title(row['SONG TITLE'])[:7]
        if key7 in song_norm7_to_isrc:
            rows_filled.at[index, 'ISRC'] = song_norm7_to_isrc[key7]
            rows_filled.at[index, 'Song'] = song_norm7_to_song[key7]
    return rows_filled


before_7char_pass = adapted_df.copy()
adapted_df = fill_rows_ISRC_song_7char(adapted_df)

print("\nSecond attempt (first 7 characters of normalized song title) fill results:")
count_filled(before_7char_pass, adapted_df, columns=('ISRC', 'Song'))


def fill_rows_ISRC_song_ignore_cantonese(df):
    """Attempt: ignore the '粵' (Cantonese) marker in the song title before matching."""
    rows_filled = df.copy()
    for index, row in rows_filled.iterrows():
        if row['ISRC'] != 'XX_UNKNOWN':
            continue
        norm_song = normalize_catalog_title(row['SONG TITLE']).replace('粵', '')
        if norm_song in song_norm_to_isrc:
            rows_filled.at[index, 'ISRC'] = song_norm_to_isrc[norm_song]
            rows_filled.at[index, 'Song'] = song_norm_to_song[norm_song]
    return rows_filled


before_cantonese_pass = adapted_df.copy()
adapted_df = fill_rows_ISRC_song_ignore_cantonese(adapted_df)

print("\nCantonese marker (粵) suffix pass fill results:")
count_filled(before_cantonese_pass, adapted_df, columns=('ISRC', 'Song'))

check_cols = ['ISRC', 'Song', 'Album', 'Album Type']
unfilled_mask = adapted_df[check_cols].isna() | adapted_df[check_cols].isin(UNFILLED_MARKERS)
rows_still_unfilled = adapted_df[unfilled_mask.any(axis=1)]
print(f"Rows with at least one unfilled value in {check_cols}: {len(rows_still_unfilled):,}")

if len(rows_still_unfilled) > 0:
    unfilled_outputfile = 'Rock_royalties_2021Q1_2026Q1_2_unfilled_rows.xlsx'
    unfilled_path = os.path.join(outputdirectory, unfilled_outputfile)
    rows_still_unfilled.to_excel(unfilled_path, index=False, engine='openpyxl')
    print(f"Wrote {len(rows_still_unfilled):,} rows to {unfilled_path}")

rows_still_unfilled.head(50)



ISRC / Song fill results:
Unmapped rows: 34,240
Values filled by algorithm:
  ISRC: 33,501 (97.8%)
  Song: 33,501 (97.8%)
Rows with filled value in ['ISRC', 'Song']: 33,501 (97.8%)

Album / Album Type fill results:
Unmapped rows: 34,240
Values filled by algorithm:
  Album: 34,240 (100.0%)
  Album Type: 34,240 (100.0%)
Rows with filled value in ['Album', 'Album Type']: 34,240 (100.0%)

Lyric-video (歌詞版MV) suffix pass fill results:
Unmapped rows: 739
Values filled by algorithm:
  ISRC: 144 (19.5%)
  Song: 144 (19.5%)
Rows with filled value in ['ISRC', 'Song']: 144 (19.5%)

Second attempt (first 7 characters of normalized song title) fill results:
Unmapped rows: 595
Values filled by algorithm:
  ISRC: 362 (60.8%)
  Song: 362 (60.8%)
Rows with filled value in ['ISRC', 'Song']: 362 (60.8%)

Cantonese marker (粵) suffix pass fill results:
Unmapped rows: 233
Values filled by algorithm:
  ISRC: 233 (100.0%)
  Song: 233 (100.0%)
Rows with filled value in ['ISRC', 'Song']: 233 (100.0%)
Rows with

,AMOUNT,ARTIST,Base Price,CATALOG NO.,CATALOG NO._MOD,CATALOG TITLE,Ctrl.%,Currency,Entry No.,PayType,...,AMOUNT (HKD),Amount to Rock (HKD),FX rate_rmb,Amount to Rock (RMB),CATALOG NO..CATALOG TITLE.SONG TITLE,CATALOG NO..CATALOG TITLE.SONG TITLE_MOD,ISRC,Song,Album,Album Type


In [94]:
def is_problem_value(series):
    as_str = series.astype(str).str.strip()
    return (
        series.isna()
        | as_str.eq('')
        | as_str.str.upper().eq('NAN')
        | as_str.eq('XX_UNKNOWN')
    )

check_cols_final = ['ISRC', 'Song', 'Album', 'Album Type']
problem_mask = adapted_df[check_cols_final].apply(is_problem_value)
rows_with_problems = adapted_df[problem_mask.any(axis=1)]

cols_with_problems = [col for col in check_cols_final if problem_mask[col].any()]
print(f"Rows with 'XX_UNKNOWN', blank, or 'NAN' in {check_cols_final}: {len(rows_with_problems):,}")
print(f"Columns affected: {cols_with_problems}")
for col in check_cols_final:
    print(f"  {col}: {problem_mask[col].sum():,} cells")

rows_with_problems

# ---------- Write out the final matched file, now that all matching (incl. the fallback passes above) is done ----------

# Fold the improved ISRC / Song / Album / Album Type values (found via the fallback passes on the
# originally-unmapped rows) back into the full matched dataframe before exporting.
fill_cols = ['ISRC', 'Song', 'Album', 'Album Type']
df_Rock_matched.loc[adapted_df.index, fill_cols] = adapted_df[fill_cols]

# Safety-net fillna, in case any rows still have missing values after all matching passes above
df_Rock_matched.fillna({'Song':'EMPTY','Album':'XX_UNKNOWN','Album Type':'XX_UNKNOWN'}, inplace=True)

print(f"The dataframe now has {df_Rock_matched.shape[0]} rows and {len(df_Rock_matched.columns)} columns.")

for column in df_Rock_matched.columns:        
        print(column)

path = os.path.join(outputdirectory, outputfile)
df_Rock_matched.to_csv(path, index=False)
print(f"\nSaved final matched file ({df_Rock_matched.shape[0]:,} rows, {df_Rock_matched.shape[1]} columns) to: {path}")

Rows with 'XX_UNKNOWN', blank, or 'NAN' in ['ISRC', 'Song', 'Album', 'Album Type']: 0
Columns affected: []
  ISRC: 0 cells
  Song: 0 cells
  Album: 0 cells
  Album Type: 0 cells
The dataframe now has 1442752 rows and 37 columns.
AMOUNT
AMOUNT (HKD)
ARTIST
Album
Album Type
Amount to Rock (HKD)
Amount to Rock (RMB)
Base Price
CATALOG NO.
CATALOG TITLE
Ctrl.%
Currency
Entry No.
FX rate
FX rate_rmb
ISRC
PayType
Payee/Licensor
Payer/Licensee
REVENUE PERIOD
ROYALTY
Release Date
Report Quarter
Report year
Rev Quarter
Rev Year
Royalty Rate%
SHARE AMOUNT (local FX)
SONG TITLE
Share%
Song
SongProRata
Territory
Type
UNIT
USER
WS Price

Saved final matched file (1,442,752 rows, 37 columns) to: ../../50 KM Group/Royalties/Statements/Karen/_output/Rock_royalties_2021Q1_2026Q1_2_matched.csv
